In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [6]:
import numpy as np
import pandas as pd
import sys
sys.path.append('/content/drive/MyDrive/EPFL/ml4bd/project')
from preprocess import *
from utils import *


import torch
import torch.nn as nn

## Data preprocessing

In [7]:
DATA_DIR = './drive/MyDrive/EPFL/ml4bd/project/data'

In [8]:
users_clean, events_clean, transactions_clean, event_users_features, feature_cols, df_full = preprocess(DATA_DIR)

In [ ]:
print(feature_cols)

## Experimental setups

In [11]:
# device setup :
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


### Basic setup

We take all time data and no demographic informations.

In [ ]:
train_loader, val_loader, test_loader = create_dataset(df_full, feature_cols, users_clean, demographics=False, SEQUENCE_LENGTH=12, mode="default")

In [ ]:
basic_model = LSTMModel(input_size=len(feature_cols)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(basic_model.parameters(), lr=1e-3)

EPOCHS = 30
patience = 5  # early stopping patience

model = train_model_no_demographics(basic_model, train_loader, val_loader, EPOCHS, device, criterion, optimizer, patience)

evaluate_model(model, test_loader, device)


### Test without summer

In this subsection, we discard summer data in the test set. In fact it's very easy for the model to learn that in summer there is no dropout, so it boosts performances when we look at the metrics.

In [ ]:
train_loader, val_loader, test_loader = create_dataset(df_full, feature_cols, users_clean, demographics=False, SEQUENCE_LENGTH=12, mode="drop_summer_test")

In [ ]:
without_summer_model = LSTMModel(input_size=len(feature_cols)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(without_summer_model.parameters(), lr=1e-3)

EPOCHS = 30
patience = 5  # early stopping patience

without_summer_model = train_model_no_demographics(without_summer_model, train_loader, val_loader, EPOCHS, device, criterion, optimizer, patience)

evaluate_model(without_summer_model, test_loader, device)

Epoch 1 | Train Loss: 1018.2063 | Val Loss: 251.1606 | Val F1: 0.8049
✅ New best model!
Epoch 2 | Train Loss: 954.4319 | Val Loss: 247.9622 | Val F1: 0.8053
✅ New best model!
Epoch 3 | Train Loss: 936.9819 | Val Loss: 241.6267 | Val F1: 0.8117
✅ New best model!
Epoch 4 | Train Loss: 918.9453 | Val Loss: 240.6341 | Val F1: 0.8138
✅ New best model!
Epoch 5 | Train Loss: 906.2903 | Val Loss: 238.6792 | Val F1: 0.8042
Epoch 6 | Train Loss: 893.8287 | Val Loss: 237.3918 | Val F1: 0.8129
Epoch 7 | Train Loss: 883.5700 | Val Loss: 238.6588 | Val F1: 0.8103
Epoch 8 | Train Loss: 871.6265 | Val Loss: 240.6297 | Val F1: 0.8106
Epoch 9 | Train Loss: 861.6254 | Val Loss: 235.4722 | Val F1: 0.8117
⏹ Early stopping triggered
Best Validation F1: 0.8138
ROC-AUC: 0.8191853819332278
F1: 0.8153505535055351
Precision: 0.7417154519576776
Recall: 0.9052176192973257
Best F1: 0.8179445894175782
Best threshold: 0.3938775510204082


### Model with only time features

Interruptions in the use is highly correlated with the time of the year : we expect student to use less the tool in september when they just started class again, and around vacations during year for example. In this section we use only time features (year and day) to see how well we can predict such interruptions without knowing anything about the student.

In [8]:
time_features = ["year", "day"]
train_loader, val_loader, test_loader = create_dataset(df_full, time_features, users_clean, demographics=False, SEQUENCE_LENGTH=12, mode="default")

In [ ]:
only_time_model = LSTMModel(input_size=len(time_features)).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(only_time_model.parameters(), lr=1e-3)

EPOCHS = 30
patience = 5  # early stopping patience

only_time_model = train_model_no_demographics(only_time_model, train_loader, val_loader, EPOCHS, device, criterion, optimizer, patience)

evaluate_model(only_time_model, test_loader, device)

Epoch 1 | Train Loss: 1082.8730 | Val Loss: 275.2544 | Val F1: 0.7656
✅ New best model!
Epoch 2 | Train Loss: 1044.5048 | Val Loss: 272.3021 | Val F1: 0.7702
✅ New best model!
Epoch 3 | Train Loss: 1038.9708 | Val Loss: 293.6350 | Val F1: 0.7587
Epoch 4 | Train Loss: 1035.6081 | Val Loss: 273.0423 | Val F1: 0.7717
✅ New best model!
Epoch 5 | Train Loss: 1027.4966 | Val Loss: 266.2055 | Val F1: 0.7914
✅ New best model!
Epoch 6 | Train Loss: 1020.2312 | Val Loss: 267.5274 | Val F1: 0.7940
✅ New best model!
Epoch 7 | Train Loss: 1016.6887 | Val Loss: 269.4639 | Val F1: 0.7921
Epoch 8 | Train Loss: 1015.2020 | Val Loss: 265.3215 | Val F1: 0.7948
✅ New best model!
Epoch 9 | Train Loss: 1012.8990 | Val Loss: 266.1267 | Val F1: 0.7897
Epoch 10 | Train Loss: 1010.0540 | Val Loss: 264.7603 | Val F1: 0.7952
✅ New best model!
Epoch 11 | Train Loss: 1010.3798 | Val Loss: 266.6091 | Val F1: 0.7974
✅ New best model!
Epoch 12 | Train Loss: 1010.0723 | Val Loss: 266.3759 | Val F1: 0.7959
Epoch 13 | Tr

### Adding demographic features

In [9]:
train_loader, val_loader, test_loader = create_dataset(df_full, feature_cols, users_clean, demographics=True, SEQUENCE_LENGTH=12, mode="default")

In [12]:
demo_model = LSTMWithDemo(
    input_size=len(feature_cols),
    n_gender=4,
    n_canton=25,
    n_class=85
).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(demo_model.parameters(), lr=1e-3)

EPOCHS = 30
patience = 5  # early stopping patience

demo_model = train_model_demographics(demo_model, train_loader, val_loader, EPOCHS, device, criterion, optimizer, patience)

evaluate_model_demographics(demo_model, test_loader, device)

Epoch 01 | Train Loss: 0.4412 | Val Loss: 0.4395 | Val F1: 0.7768
✅ New best model!
Epoch 02 | Train Loss: 0.4141 | Val Loss: 0.4329 | Val F1: 0.7831
✅ New best model!
Epoch 03 | Train Loss: 0.4005 | Val Loss: 0.4266 | Val F1: 0.7981
✅ New best model!
Epoch 04 | Train Loss: 0.3923 | Val Loss: 0.3920 | Val F1: 0.8120
✅ New best model!
Epoch 05 | Train Loss: 0.3820 | Val Loss: 0.3838 | Val F1: 0.8265
✅ New best model!
Epoch 06 | Train Loss: 0.3747 | Val Loss: 0.3766 | Val F1: 0.8284
✅ New best model!
Epoch 07 | Train Loss: 0.3663 | Val Loss: 0.3734 | Val F1: 0.8215
Epoch 08 | Train Loss: 0.3606 | Val Loss: 0.3764 | Val F1: 0.8229
Epoch 09 | Train Loss: 0.3539 | Val Loss: 0.3914 | Val F1: 0.8176
Epoch 10 | Train Loss: 0.3479 | Val Loss: 0.3656 | Val F1: 0.8276
Epoch 11 | Train Loss: 0.3418 | Val Loss: 0.3681 | Val F1: 0.8274
⏹ Early stopping triggered
Best Validation F1: 0.8284
